# 2. Tendencia lineal + variables indicadoras del mes

In [1]:
%run functions.ipynb

In [2]:
from sklearn.linear_model import LinearRegression

df = load_data()
df = add_linear_trend_component(df)
df = add_month_component(df)

# Una columna indicadora por mes (se omite enero para evitar colinealidad)
dummies = pd.get_dummies(df.month, prefix="month", drop_first=True, dtype=int)
df = pd.concat([df, dummies], axis=1)

x_columns = ["trend"] + list(dummies.columns)
X_complete, y_complete, X_train, y_train, X_test, y_test = train_test_split(df, x_columns, "yt_true")

model = LinearRegression().fit(X_train, y_train)
df["yt_pred_trend_month"] = model.predict(X_complete)

plot_time_series(df[["yt_true", "yt_pred_trend_month"]])

<Figure size 1200x400 with 1 Axes>

In [3]:
metrics = compute_evaluation_metrics(df)
save_forecasts(df)
save_metrics(metrics)
metrics

,Metrics,yt_pred_trend_month
0,MSE Train,103217.67
1,MSE Test,738282.04
2,MAE Train,259.03
3,MAE Test,804.13


El ajuste en entrenamiento mejora mucho, pero en prueba no: el modelo es aditivo (la misma oscilacion en pesos todos los anos) mientras que la oscilacion real crece con el nivel. Ademas la tendencia real se acelera al final y la recta se queda corta.